In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)


TRAIN_PATH = "../data/processed/train_table_train_2015_2024.parquet"
TEST_PATH  = "../data/processed/train_table_test_2025.parquet"
OUT_DIR    = "../models"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42


train_table = pd.read_parquet(TRAIN_PATH)
test_table  = pd.read_parquet(TEST_PATH)


train_table["time_bin"] = pd.to_datetime(train_table["time_bin"])
test_table["time_bin"]  = pd.to_datetime(test_table["time_bin"])


train_df = train_table[train_table["time_bin"].dt.year <= 2023].copy()
val_df   = train_table[train_table["time_bin"].dt.year == 2024].copy()
test_df  = test_table.copy()  

print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
print("y mean:", train_df["y"].mean(), val_df["y"].mean(), test_df["y"].mean())


drop_cols = ["y", "time_bin"]  
X_train = train_df.drop(columns=drop_cols)
y_train = train_df["y"].astype(int)

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["y"].astype(int)

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["y"].astype(int)

cat_cols = ["cell_id"]  # categorical
num_cols = [c for c in X_train.columns if c not in cat_cols]


preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
        ("num", StandardScaler(with_mean=False), num_cols),  # with_mean=False for sparse safety
    ],
    remainder="drop",
)

clf = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    n_jobs=None,
    random_state=RANDOM_STATE,
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", clf),
])


pipe.fit(X_train, y_train)


def eval_split(name, X, y):
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    ap  = average_precision_score(y, proba)
    print(f"\n[{name}] AUC={auc:.4f}  AP={ap:.4f}")
    print("Confusion:\n", confusion_matrix(y, pred))
    print(classification_report(y, pred, digits=4))

eval_split("VAL(2024)", X_val, y_val)
eval_split("TEST(2025)", X_test, y_test)


joblib.dump(pipe, os.path.join(OUT_DIR, "logreg_grid.pkl"))
print("\nSaved:", os.path.join(OUT_DIR, "logreg_grid.pkl"))

Train/Val/Test: (292810, 15) (33019, 15) (31773, 15)
y mean: 0.5859157815648373 0.6036221569399437 0.570264060680452

[VAL(2024)] AUC=0.8589  AP=0.9007
Confusion:
 [[ 9011  4077]
 [ 3060 16871]]
              precision    recall  f1-score   support

           0     0.7465    0.6885    0.7163     13088
           1     0.8054    0.8465    0.8254     19931

    accuracy                         0.7839     33019
   macro avg     0.7759    0.7675    0.7709     33019
weighted avg     0.7820    0.7839    0.7822     33019


[TEST(2025)] AUC=0.8571  AP=0.8853
Confusion:
 [[ 9253  4401]
 [ 2639 15480]]
              precision    recall  f1-score   support

           0     0.7781    0.6777    0.7244     13654
           1     0.7786    0.8544    0.8147     18119

    accuracy                         0.7784     31773
   macro avg     0.7784    0.7660    0.7696     31773
weighted avg     0.7784    0.7784    0.7759     31773


Saved: ../models/logreg_grid.pkl


/Users/yueyanghe/Desktop/milestone2/Team9_IT5006_Predictive_Policing_AY2526Sem2/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
